# SQLAlchemy Core

**SQLAlchemy** — это программная библиотека на Python для работы с реляционными СУБД.
SQLAlchemy состоит из двух частей:

* **Core** — низкоуровневый API: работа со схемой таблиц, генерация SQL-выражений, прямое выполнение запросов через `Connection`.
* **ORM** — высокоуровневая надстройка над Core, использующая классы Python для представления строк.

В данном ноутбуке рассматривается **SQLAlchemy Core** — подход, при котором мы описываем таблицы явно через объекты `Table` и `Column`, а запросы строим через конструкторы `select()`, `insert()`, `update()`, `delete()`.

### Преимущества Core
* Полный контроль над SQL-запросами.
* Меньше «магии» — выше прозрачность.
* Ниже накладные расходы по сравнению с ORM.
* Идеально подходит для сложных аналитических запросов и пакетных операций.


## Архитектура SQLAlchemy

```SQLAlchemy``` предоставляет богатый API для каждого уровня взаимодействия с БД, разбивая общую задачу на две категории: **ядро (Core)** и объектно-реляционное представление (ORM).

Core включает:
* взаимодействие с Python DB-API;
* обработку текстовых SQL-запросов;
* управление схемой БД через объект `MetaData` и классы `Table` / `Column`.

Основные объекты Core:
| Объект | Назначение |
|---|---|
| `create_engine()` | Создаёт «двигатель» подключения к БД |
| `MetaData` | Хранилище описаний всех таблиц |
| `Table` | Описание таблицы |
| `Column` | Описание колонки |
| `Connection` | Контекстный менеджер для выполнения запросов |
| `select / insert / update / delete` | Конструкторы DML-запросов |


### Установка SQLAlchemy

Установка стандартная:

```
pip install SQLAlchemy
```

Проверка версии:
```python
import sqlalchemy
print("Версия SQLAlchemy:", sqlalchemy.__version__)
```


In [ ]:
import sqlalchemy
print("Версия SQLAlchemy:", sqlalchemy.__version__)

## Создание схемы в SQLAlchemy Core

В `SQLAlchemy Core` таблицы описываются явно с помощью объектов `Table` и `Column`, объединённых в общий `MetaData`. Это не классы-модели, а структуры данных, которые SQLAlchemy использует для генерации SQL.

Основные шаги:
1. Создать `engine` (подключение к СУБД).
2. Создать объект `MetaData`.
3. Описать таблицы через `Table(...)`.
4. Вызвать `metadata.create_all(engine)` для создания таблиц в БД.


### Создание таблицы `student`

Опишем таблицу студентов с полями: `id`, `surname`, `name`, `patronymic`, `age`, `address`.


In [ ]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String

DATABASE_NAME = 'student.sqlite'

engine = create_engine(f'sqlite:///{DATABASE_NAME}')
metadata = MetaData()

In [ ]:
student_table = Table(
    'student', metadata,
    Column('id',         Integer, primary_key=True, autoincrement=True),
    Column('surname',    String),
    Column('name',       String),
    Column('patronymic', String),
    Column('age',        Integer),
    Column('address',    String),
)

In [ ]:
# Создаём таблицы в БД
metadata.create_all(engine)
print("Таблица 'student' создана.")

### Заполнение фиктивными данными

Для вставки данных используем конструктор `insert()` и метод `conn.execute()`.


In [ ]:
from faker import Faker
from sqlalchemy import insert

def load_fake_student_data(engine, table, n=50):
    faker = Faker('ru_RU')
    rows = []
    for _ in range(n):
        parts = faker.name().split(' ')
        # Faker иногда возвращает 2 или 4 части — нормализуем до 3
        while len(parts) < 3:
            parts.append('')
        rows.append({
            'surname':    parts[0],
            'name':       parts[1],
            'patronymic': parts[2],
            'age':        faker.random.randint(18, 23),
            'address':    faker.address(),
        })

    with engine.connect() as conn:
        conn.execute(insert(table), rows)
        conn.commit()
    print(f"Вставлено {n} студентов.")

load_fake_student_data(engine, student_table)


### Запросы

В Core для выборки данных используется конструктор `select()`.
Результат выполнения — объект `CursorResult`, по которому можно итерироваться.


In [ ]:
from sqlalchemy import select, func

# Количество студентов
with engine.connect() as conn:
    count = conn.execute(select(func.count()).select_from(student_table)).scalar()
    print('*' * 30)
    print(f'Кол-во студентов: {count}')
    print('*' * 30)

In [ ]:
# Получить студента по id = 40
with engine.connect() as conn:
    stmt = select(student_table).where(student_table.c.id == 40)
    row = conn.execute(stmt).fetchone()
    print(row)

In [ ]:
# Все студенты
with engine.connect() as conn:
    result = conn.execute(select(student_table))
    for row in result:
        print(row)

In [ ]:
# Студенты в возрасте от 20 до 22 лет
from sqlalchemy import and_

with engine.connect() as conn:
    stmt = select(student_table).where(
        and_(student_table.c.age >= 20, student_table.c.age <= 22)
    )
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Выбрать только ФИО — select нужных колонок
with engine.connect() as conn:
    stmt = select(
        student_table.c.surname,
        student_table.c.name,
        student_table.c.patronymic,
        student_table.c.age
    ).order_by(student_table.c.surname)
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# LIMIT / OFFSET
with engine.connect() as conn:
    stmt = select(student_table).limit(5).offset(10)
    for row in conn.execute(stmt):
        print(row)

### Обновление и удаление

В Core используются конструкторы `update()` и `delete()`.


In [ ]:
from sqlalchemy import update, delete

# Обновить адрес студента с id=1
with engine.connect() as conn:
    stmt = (
        update(student_table)
        .where(student_table.c.id == 1)
        .values(address='Санкт-Петербург, Невский проспект, 1')
    )
    conn.execute(stmt)
    conn.commit()
    row = conn.execute(select(student_table).where(student_table.c.id == 1)).fetchone()
    print("После обновления:", row)

In [ ]:
# Удалить студентов моложе 19 лет
with engine.connect() as conn:
    before = conn.execute(select(func.count()).select_from(student_table)).scalar()
    conn.execute(delete(student_table).where(student_table.c.age < 19))
    conn.commit()
    after = conn.execute(select(func.count()).select_from(student_table)).scalar()
    print(f"До удаления: {before}, после: {after}")

## Связанные таблицы: студенты, группы и предметы

Теперь рассмотрим более сложную схему с тремя таблицами и связями между ними:

* `student` — студенты (many-to-one → groups)
* `groups` — учебные группы
* `lessons` — предметы
* `association` — таблица связи many-to-many между `lessons` и `groups`

В Core внешние ключи и таблицы связи описываются явно через `ForeignKey`.


In [ ]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, ForeignKey

DATABASE_NAME_2 = 'student_group_lesson.sqlite'

engine2 = create_engine(f'sqlite:///{DATABASE_NAME_2}')
metadata2 = MetaData()

In [ ]:
groups_table = Table(
    'groups', metadata2,
    Column('id',         Integer, primary_key=True, autoincrement=True),
    Column('group_name', String, nullable=False),
)

student_table2 = Table(
    'student', metadata2,
    Column('id',         Integer, primary_key=True, autoincrement=True),
    Column('surname',    String),
    Column('name',       String),
    Column('patronymic', String),
    Column('age',        Integer),
    Column('address',    String),
    Column('group_id',   Integer, ForeignKey('groups.id')),
)

# Таблица связи many-to-many: предмет <-> группа
association_table = Table(
    'association', metadata2,
    Column('lesson_id', Integer, ForeignKey('lessons.id')),
    Column('group_id',  Integer, ForeignKey('groups.id')),
)

lessons_table = Table(
    'lessons', metadata2,
    Column('id',           Integer, primary_key=True, autoincrement=True),
    Column('lesson_title', String, nullable=False),
)

In [ ]:
metadata2.create_all(engine2)
print("Таблицы созданы.")

### Заполнение тестовыми данными

Добавим группы, предметы, связи между ними, а затем студентов.


In [ ]:
from faker import Faker
from sqlalchemy import insert

def load_fake_data(engine, g_tbl, s_tbl, a_tbl, l_tbl):
    lessons_names = [
        'Экономика и менеджмент горного производства',
        'Аэрология горных предприятий',
        'Методы математической обработки маркшейдерско-геодезических измерений',
        'Алгоритмы и программы автоматизации маркшейдерско-геодезических работ',
        'Маркшейдерское обеспечение подземного строительства',
        'Информационное обеспечение маркшейдерских работ',
        'Маркшейдерско-геомеханическое обеспечение безопасности горных работ',
        'Геометрия недр',
        'Дистанционные методы съемок в маркшейдерском обеспечении',
        'Основы конструирования измерительных систем',
    ]

    faker = Faker('ru_RU')

    with engine.connect() as conn:
        # Группы
        result1 = conn.execute(insert(g_tbl).values(group_name='ГГ-22-1'))
        result2 = conn.execute(insert(g_tbl).values(group_name='ГГ-22-2'))
        g1_id = result1.inserted_primary_key[0]
        g2_id = result2.inserted_primary_key[0]

        # Предметы и связи
        for idx, title in enumerate(lessons_names):
            res = conn.execute(insert(l_tbl).values(lesson_title=title))
            l_id = res.inserted_primary_key[0]
            conn.execute(insert(a_tbl).values(lesson_id=l_id, group_id=g1_id))
            if idx % 2 == 0:
                conn.execute(insert(a_tbl).values(lesson_id=l_id, group_id=g2_id))

        # Студенты
        group_ids = [g1_id, g2_id]
        student_rows = []
        for _ in range(50):
            parts = faker.name().split(' ')
            while len(parts) < 3:
                parts.append('')
            student_rows.append({
                'surname':    parts[0],
                'name':       parts[1],
                'patronymic': parts[2],
                'age':        faker.random.randint(16, 25),
                'address':    faker.address(),
                'group_id':   faker.random.choice(group_ids),
            })
        conn.execute(insert(s_tbl), student_rows)
        conn.commit()

    print("Данные загружены.")

load_fake_data(engine2, groups_table, student_table2, association_table, lessons_table)


### Запросы

#### Все предметы


In [ ]:
from sqlalchemy import select

with engine2.connect() as conn:
    for row in conn.execute(select(lessons_table)):
        print(row)

In [ ]:
# Предметы с id > 3
with engine2.connect() as conn:
    stmt = select(lessons_table).where(lessons_table.c.id > 3)
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Предметы, название которых начинается на 'М' или 'Д'
from sqlalchemy import or_

with engine2.connect() as conn:
    stmt = select(lessons_table).where(
        or_(
            lessons_table.c.lesson_title.like('М%'),
            lessons_table.c.lesson_title.like('Д%'),
        )
    )
    for row in conn.execute(stmt):
        print(row)

#### Декартово произведение таблиц (без JOIN)

В Core можно перечислить несколько таблиц в `select()` — получим все комбинации строк.


In [ ]:
# Декартово произведение lessons × groups (аналог ORM: query(Lesson, Group))
with engine2.connect() as conn:
    stmt = select(lessons_table, groups_table)
    for row in conn.execute(stmt):
        print(row)

#### JOIN через таблицу связи

Соединим `lessons` и `groups` через `association`, используя явный `WHERE`.


In [ ]:
from sqlalchemy import and_

with engine2.connect() as conn:
    stmt = select(lessons_table, groups_table).where(
        and_(
            association_table.c.lesson_id == lessons_table.c.id,
            association_table.c.group_id  == groups_table.c.id,
        )
    )
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Только для группы ГГ-22-2
with engine2.connect() as conn:
    stmt = select(lessons_table, groups_table).where(
        and_(
            association_table.c.lesson_id == lessons_table.c.id,
            association_table.c.group_id  == groups_table.c.id,
            groups_table.c.group_name     == 'ГГ-22-2',
        )
    )
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Только предметы (без полей группы) для ГГ-22-2
with engine2.connect() as conn:
    stmt = select(lessons_table).where(
        and_(
            association_table.c.lesson_id == lessons_table.c.id,
            association_table.c.group_id  == groups_table.c.id,
            groups_table.c.group_name     == 'ГГ-22-2',
        )
    )
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Цепочка where() — второй фильтр добавляется отдельно
with engine2.connect() as conn:
    stmt = (
        select(lessons_table)
        .where(
            and_(
                association_table.c.lesson_id == lessons_table.c.id,
                association_table.c.group_id  == groups_table.c.id,
            )
        )
        .where(groups_table.c.group_name == 'ГГ-22-1')
    )
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Выбрать только название предмета и название группы
with engine2.connect() as conn:
    stmt = select(
        lessons_table.c.lesson_title,
        groups_table.c.group_name,
    ).where(
        and_(
            association_table.c.lesson_id == lessons_table.c.id,
            association_table.c.group_id  == groups_table.c.id,
            groups_table.c.group_name     == 'ГГ-22-2',
        )
    )
    for row in conn.execute(stmt):
        print(row)

#### JOIN с использованием метода `.join()`

В Core можно также строить запросы через явные `join()` / `join_from()`.


In [ ]:
# JOIN: предметы группы ГГ-22-2
with engine2.connect() as conn:
    stmt = (
        select(lessons_table)
        .join(association_table, association_table.c.lesson_id == lessons_table.c.id)
        .join(groups_table,      association_table.c.group_id  == groups_table.c.id)
        .where(groups_table.c.group_name == 'ГГ-22-2')
    )
    for row in conn.execute(stmt):
        print(row)

#### Студенты конкретного предмета

Найдём всех студентов, которые посещают заданный предмет, через подзапрос.


In [ ]:
# Вариант 1: последовательные запросы (аналог ORM-примера с подзапросом)
with engine2.connect() as conn:
    lesson_title = 'Основы конструирования измерительных систем'

    # Шаг 1: найти id предмета
    l_id = conn.execute(
        select(lessons_table.c.id).where(lessons_table.c.lesson_title == lesson_title)
    ).scalar()
    print("ID предмета:", l_id)

    # Шаг 2: найти id групп, связанных с предметом
    group_ids = [
        row[0] for row in conn.execute(
            select(association_table.c.group_id)
            .where(association_table.c.lesson_id == l_id)
        )
    ]
    print("ID групп:", group_ids)

    # Шаг 3: найти студентов этих групп
    stmt = select(student_table2).where(student_table2.c.group_id.in_(group_ids))
    for row in conn.execute(stmt):
        print(row)

In [ ]:
# Вариант 2: одним JOIN-запросом
with engine2.connect() as conn:
    lesson_title = 'Основы конструирования измерительных систем'

    stmt = (
        select(student_table2)
        .join(groups_table,      student_table2.c.group_id == groups_table.c.id)
        .join(association_table, association_table.c.group_id == groups_table.c.id)
        .join(lessons_table,     association_table.c.lesson_id == lessons_table.c.id)
        .where(lessons_table.c.lesson_title == lesson_title)
    )
    print(stmt)
    print("*" * 50)
    for row in conn.execute(stmt):
        print(row)

---
---
